# 3. VectorRAG: Retrieval-Augmented Generation over Chunked Text

**Part of the AlzRAGBench notebook series** (`Diffusion_AlzRAGBench_v2.0/`).

**Requires a GPU with roughly >=8GB VRAM** for the generation half of this notebook
(Section 3.6 onward). The embedding and retrieval sections (3.1-3.5) will run on CPU
too, just slower (a few minutes instead of a few seconds for embedding all chunks).

## What this notebook builds

VectorRAG is the "classic" RAG approach: embed a corpus of text chunks into a vector
space, embed the question the same way, retrieve the chunks whose embeddings are
closest to the question's embedding (cosine similarity), and hand those chunks to a
language model as context.

1. Load all 287 pre-built chunks from `../Dataset/chunking/_all_chunks.json` (created
   during dataset construction -- 20 PubMed abstracts + 10 Wikipedia articles + 1
   StatPearls textbook chapter, chunked into ~220-word windows).
2. Embed every chunk with **`NeuML/pubmedbert-base-embeddings`**, a sentence-embedding
   model fine-tuned specifically on biomedical literature -- chosen over a
   general-purpose embedding model because it should retrieve more precisely on this
   corpus of PubMed abstracts and a medical textbook chapter than a generic model
   would.
3. Build a simple **in-memory brute-force cosine-similarity index** -- deliberately
   *not* FAISS or any vector database. With only 287 chunks, a 287x768 matrix is a few
   hundred KB and a full similarity search against it is effectively instant; FAISS's
   approximate-nearest-neighbor machinery exists to make search over millions of
   vectors fast, which is not a problem we have here. Keeping this as plain NumPy also
   keeps the notebook dependency-light and easy to read line by line.
4. Load SDLM-3B-D4 (the same diffusion LLM used in Notebook 2) and generate answers
   grounded in the retrieved chunks.
5. Showcase the full pipeline on real questions from the evaluation set.

## 3.1 Install dependencies


In [ ]:
# !pip install -q torch "transformers==4.37.2" accelerate sentence-transformers numpy

## 3.2 Imports, paths, and GPU check


In [1]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import json
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer

DATASET_DIR = Path("Dataset")
CHUNKS_PATH = DATASET_DIR / "chunking" / "_all_chunks.json"
EVAL_PATH = DATASET_DIR / "Evaluation" / "eval_dataset.json"

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB VRAM)")
else:
    print("WARNING: no CUDA GPU detected. Embedding (3.3-3.4) will still run on CPU, "
          "just slower. Generation (3.6 onward) requires a GPU with roughly >=8GB VRAM.")

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


GPU: Tesla T4  (15.6 GB VRAM)


## 3.3 Load the pre-chunked corpus

`_all_chunks.json` is a flat list combining every article's chunks -- built once during
dataset construction by `Dataset/scripts/chunk_articles.py` (sentence-boundary-aware
~220-word windows with ~40-word overlap). Each chunk carries its source metadata
(`article_id`, `source_type`, `title`, etc.) alongside its text, so retrieval results
stay traceable back to a specific PubMed abstract, Wikipedia article, or textbook
section.


In [2]:
chunks = json.load(open(CHUNKS_PATH, encoding="utf-8"))
texts = [c["text"] for c in chunks]

print(f"Loaded {len(chunks)} chunks")
by_source = {}
for c in chunks:
    by_source[c["source_type"]] = by_source.get(c["source_type"], 0) + 1
print(f"By source type: {by_source}")
print(f"\nExample chunk ({chunks[0]['chunk_id']}):")
print(chunks[0]["text"][:300], "...")


Loaded 287 chunks
By source type: {'pubmed': 27, 'textbook': 43, 'wikipedia': 217}

Example chunk (pubmed_12672860__chunk000):
Title: Memantine in moderate-to-severe Alzheimer's disease. BACKGROUND: Overstimulation of the N-methyl-D-aspartate (NMDA) receptor by glutamate is implicated in neurodegenerative disorders. Accordingly, we investigated memantine, an NMDA antagonist, for the treatment of Alzheimer's disease. METHODS ...


## 3.4 Embed every chunk with a biomedical sentence-embedding model

`NeuML/pubmedbert-base-embeddings` produces 768-dimensional embeddings and was trained
on biomedical text specifically (PubMedBERT backbone + sentence-embedding fine-tuning),
which should give more precise nearest-neighbor retrieval on this corpus than a
general-purpose model like `all-MiniLM-L6-v2` would -- domain-specific vocabulary
(APOE, neurofibrillary tangles, cholinesterase inhibitors, etc.) is exactly what such a
model is tuned to represent well.

We L2-normalize embeddings at encode time so that cosine similarity reduces to a plain
dot product later.


In [3]:
EMBED_MODEL_ID = "NeuML/pubmedbert-base-embeddings"
embed_model = SentenceTransformer(EMBED_MODEL_ID, device="cuda" if torch.cuda.is_available() else "cpu")

import time
t0 = time.time()
chunk_embeddings = embed_model.encode(
    texts, batch_size=32, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True,
)
elapsed = time.time() - t0
print(f"\nEmbedded {len(texts)} chunks in {elapsed:.1f}s -> shape {chunk_embeddings.shape}")
print("(On a T4 GPU this typically takes well under 30s; on CPU, a couple of minutes.)")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]


Embedded 287 chunks in 6.9s -> shape (287, 768)
(On a T4 GPU this typically takes well under 30s; on CPU, a couple of minutes.)


## 3.5 A minimal in-memory vector index (no FAISS)

Retrieval is just: embed the query, compute cosine similarity (dot product, since
embeddings are normalized) against every chunk embedding, and take the top-k. At 287
chunks this is a single small matrix multiplication -- no approximate search structure
needed.


In [4]:
class SimpleVectorIndex:
    def __init__(self, embeddings, records, embed_model):
        self.embeddings = embeddings  # (N, D), L2-normalized
        self.records = records        # list of chunk dicts, same order as embeddings
        self.embed_model = embed_model

    def search(self, query, k=5):
        q_emb = self.embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
        sims = self.embeddings @ q_emb  # cosine similarity via dot product (both L2-normalized)
        top_idx = np.argsort(-sims)[:k]
        return [
            {**self.records[i], "score": float(sims[i])}
            for i in top_idx
        ]


vector_index = SimpleVectorIndex(chunk_embeddings, chunks, embed_model)
print(f"Vector index ready: {len(chunks)} chunks, {chunk_embeddings.shape[1]}-dim embeddings")


Vector index ready: 287 chunks, 768-dim embeddings


## 3.6 Demo retrieval

A quick sanity check before bringing in the LLM: do the top retrieved chunks actually
look relevant to a few hand-written queries?


In [5]:
def print_results(query, k=5):
    print(f"Q: {query}\n")
    for r in vector_index.search(query, k=k):
        print(f"  [{r['score']:.3f}] {r['chunk_id']} ({r['source_type']})")
        print(f"      {r['text'][:160]}...")
    print()


print_results("What is the mechanism of action of memantine?")
print_results("What genes cause familial Alzheimer's disease?")


Q: What is the mechanism of action of memantine?

  [0.652] wikipedia_memantine__chunk008 (wikipedia)
      activity as NMDA receptor antagonists. ==== Elimination ==== Memantine is eliminated primarily in urine, with 57 to 82% excreted in urine unchanged. Its elimina...
  [0.610] wikipedia_memantine__chunk004 (wikipedia)
      synapses, as it can still be activated by physiological release of glutamate following depolarization of the postsynaptic neuron. The interaction of memantine w...
  [0.609] wikipedia_memantine__chunk007 (wikipedia)
      10 to 40 mg. Peak levels after a single 20 mg dose were found to be 24 to 29 μg/L (0.13–0.16 μmol/L or μM). Steady-state levels of memantine with 20 mg/day are ...
  [0.576] wikipedia_memantine__chunk003 (wikipedia)
      excitotoxicity, is hypothesized to be involved in the etiology of Alzheimer's disease. Targeting the glutamatergic system, specifically ionotropic glutamate NMD...
  [0.561] wikipedia_memantine__chunk005 (wikipedia)
      of n

## 3.7 Load SDLM-3B-D4

Same model as Notebook 2, loaded identically -- this notebook is self-contained and
doesn't depend on Notebook 2 having been run. See Notebook 2, Sections 2.9-2.10 for a
full explanation of SDLM's semi-autoregressive block-diffusion sampling algorithm, why
`transformers` is pinned to `4.37.2`, and how the VRAM-cleanup fix works.

Section 2.9 also covers a `flash_attn` `ImportError` you may hit here even with `attn_implementation="eager"` -- it's a `transformers` import-scanner false-positive, not an actual flash_attn requirement, and the loading cell below works around it the same way Notebook 2's does.

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.dynamic_module_utils import get_imports
from unittest.mock import patch as _mock_patch

SDLM_MODEL_ID = "OpenGVLab/SDLM-3B-D4"


def _skip_flash_attn_check(filename):
    """See the flash_attn note above: modeling_sdlm.py's flash_attn import is guarded
    by `if is_flash_attn_2_available():`, but transformers' import scanner doesn't
    understand if-guards (only try/except) and flags flash_attn as required anyway,
    even though attn_implementation="eager" never touches that code path
    (github.com/huggingface/transformers/issues/28459)."""
    imports = get_imports(filename)
    if str(filename).endswith("modeling_sdlm.py") and "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports


with _mock_patch("transformers.dynamic_module_utils.get_imports", _skip_flash_attn_check):
    tokenizer = AutoTokenizer.from_pretrained(SDLM_MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        SDLM_MODEL_ID,
        trust_remote_code=True,
        attn_implementation="eager",
        torch_dtype=torch.bfloat16,
    )
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

print(f"Loaded {SDLM_MODEL_ID}")
if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loaded OpenGVLab/SDLM-3B-D4
GPU memory allocated: 7.22 GB


In [7]:
import gc

import torch
import torch.nn.functional as F


def _top_p_logits(logits, top_p):
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    sorted_indices_to_remove = cumulative_probs > top_p
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = 0
    mask = torch.zeros_like(logits, dtype=torch.bool).scatter_(-1, sorted_indices, sorted_indices_to_remove)
    return logits.masked_fill(mask, torch.finfo(logits.dtype).min)


def _sample_tokens(logits, temperature=0.0, top_p=None, entropy_conf=False):
    """Returns (confidence, token_id) for each of the n_future_tokens candidate
    positions. Confidence is either top-probability (used to decide how many of the
    candidate tokens to "accept" this step) or a normalized-entropy score."""
    if temperature > 0:
        logits = logits / temperature
    if top_p is not None and top_p < 1:
        logits = _top_p_logits(logits, top_p)
    probs = torch.softmax(logits, dim=-1)

    if temperature > 0:
        x0 = torch.distributions.Categorical(probs=probs).sample()
        confidence = torch.gather(probs, -1, x0.unsqueeze(-1)).squeeze(-1)
    else:
        confidence, x0 = probs.max(dim=-1)

    if entropy_conf:
        entropy = -torch.sum(probs * torch.log(probs + 1e-9), dim=-1)
        max_entropy = torch.log(torch.tensor(float(probs.shape[-1]), device=entropy.device))
        confidence = 1.0 - (entropy / max_entropy)

    return confidence, x0


def _find_accept_length(confidence, threshold):
    """How many of the n_future_tokens candidate tokens (in order) are confident
    enough to commit this step -- SDLM accepts the longest confident *prefix*, since
    later candidate positions were predicted assuming all earlier ones land correctly."""
    mask = confidence > threshold
    first_low_confidence = torch.argmax((~mask).int(), dim=1)
    all_confident = mask.all(dim=1)
    seq_len = confidence.shape[1]
    return torch.where(all_confident, torch.full_like(first_low_confidence, seq_len), first_low_confidence)


@torch.no_grad()
def sdlm_generate(model, tokenizer, input_ids, max_gen_len=192, threshold=0.5,
                   n_future_tokens=4, temperature=0.0, top_p=None, alg="prob_conf"):
    """Semi-autoregressive block-diffusion generation with KV-cache reuse. Adapted from
    OpenGVLab's official SDLM_generate (github.com/OpenGVLab/SDLM), trimmed to the
    prob_conf/entropy_conf paths, with explicit intermediate-tensor cleanup added each
    iteration (see Section 2.10 markdown for why that matters).

    At each step: append n_future_tokens mask-token placeholders after the current
    sequence, run one forward pass reusing the cached keys/values from previous steps,
    and predict all n_future_tokens positions at once. Accept however many of those (in
    order, starting from the first) the model is confident enough about, roll the KV
    cache back to just past the accepted tokens, and repeat.
    """
    device = model.device
    mask_token_id = getattr(model.config, "text_mask_token_id", 151665)
    eos_ids = set(tokenizer.eos_token_id) if isinstance(tokenizer.eos_token_id, (list, tuple)) else {tokenizer.eos_token_id}
    gen_eos = getattr(model.generation_config, "eos_token_id", None)
    if gen_eos is not None:
        eos_ids |= set(gen_eos) if isinstance(gen_eos, (list, tuple)) else {gen_eos}

    model.model.block_size = getattr(model.config, "block_size", n_future_tokens)
    model.model.causal_attn = getattr(model.config, "causal_attn", False)
    model.model.text_mask_token_id = mask_token_id
    model.use_cache = True

    generated = input_ids.clone()
    prompt_len = generated.shape[1]
    target_len = prompt_len + max_gen_len
    past_key_values = None

    while generated.shape[1] < target_len:
        mask_tokens = torch.full((1, n_future_tokens - 1), mask_token_id, dtype=generated.dtype, device=device)
        generated_with_mask = torch.cat([generated, generated[:, -1:], mask_tokens], dim=1)

        start_idx = past_key_values[0][0].shape[2] if past_key_values is not None else 0
        position_ids = torch.arange(start_idx, generated_with_mask.shape[1], device=device).unsqueeze(0)
        position_ids[0, -n_future_tokens:] -= 1

        model_inputs = model.prepare_inputs_for_generation(
            generated_with_mask, past_key_values, None, use_cache=True, position_ids=position_ids,
        )
        outputs = model(**model_inputs)

        logits = outputs.logits[:, -n_future_tokens:, :]
        confidence, x0 = _sample_tokens(logits, temperature=temperature, top_p=top_p, entropy_conf=(alg == "entropy_conf"))
        accept_len = max(1, min(int(_find_accept_length(confidence, threshold)[0].item()), n_future_tokens))

        new_tokens = x0[0, :accept_len]
        eos_pos = next((i for i, t in enumerate(new_tokens.tolist()) if t in eos_ids), None)
        if eos_pos is not None:
            new_tokens = new_tokens[:eos_pos]

        generated = torch.cat([generated, new_tokens.unsqueeze(0)], dim=1)

        # Roll the KV cache back to only the tokens actually committed this step --
        # this, plus the cleanup below, is what keeps VRAM flat across many calls
        # instead of creeping upward the way the previous (LLaDA) notebook's
        # full-resequence-every-step approach did.
        past_key_values = tuple(
            (k[:, :, : generated.shape[1], :], v[:, :, : generated.shape[1], :])
            for k, v in outputs.past_key_values
        )
        del outputs, logits, confidence, x0, generated_with_mask, model_inputs

        if eos_pos is not None:
            break

    return generated[:, prompt_len:]


def generate_with_sdlm(prompt_text, system_prompt=None, max_gen_len=192, threshold=0.5,
                        n_future_tokens=4, temperature=0.0):
    """High-level helper: apply the chat template, run sdlm_generate, decode the
    response, and release GPU memory before returning -- see Section 2.10."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt_text})

    formatted = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    input_ids = tokenizer([formatted], return_tensors="pt").input_ids.to(model.device)

    response_ids = sdlm_generate(
        model, tokenizer, input_ids,
        max_gen_len=max_gen_len, threshold=threshold, n_future_tokens=n_future_tokens, temperature=temperature,
    )
    answer = tokenizer.decode(response_ids[0], skip_special_tokens=True).strip()

    del input_ids, response_ids
    gc.collect()
    torch.cuda.empty_cache()

    return answer

## 3.8 VectorRAG prompt template and end-to-end answer function

Retrieved chunks are concatenated (with their source citations) into a context block,
and the model is instructed to answer using only that context.


In [8]:
VECTOR_SYSTEM_PROMPT = (
    "You are a biomedical question-answering assistant. You will be given several "
    "text passages retrieved from Alzheimer's disease research articles and a "
    "textbook. Answer the user's question using only information from these "
    "passages. Be concise and directly address the question."
)


def build_vector_prompt(question, retrieved_chunks):
    context = "\n\n".join(
        f"[Passage from {r['article_id']}]\n{r['text']}" for r in retrieved_chunks
    )
    return f"Retrieved passages:\n{context}\n\nQuestion: {question}"


def vectorrag_answer(question, k=5):
    retrieved = vector_index.search(question, k=k)
    prompt = build_vector_prompt(question, retrieved)
    answer = generate_with_sdlm(prompt, system_prompt=VECTOR_SYSTEM_PROMPT)
    return {"question": question, "retrieved": retrieved, "answer": answer}

## 3.9 Showcase: VectorRAG on real evaluation questions

We run the full pipeline on a few questions from `../Dataset/Evaluation/eval_dataset.json`
that were specifically designed to favor VectorRAG -- single-hop facts that should be
directly stated within one chunk, without needing to connect information across
multiple sources.


In [9]:
eval_data = json.load(open(EVAL_PATH, encoding="utf-8"))
questions_by_id = {q["question_id"]: q for q in eval_data["questions"]}


def show_result(result, gold_answer):
    print(f"Question: {result['question']}\n")
    print("Retrieved chunks:")
    for r in result["retrieved"]:
        print(f"  [{r['score']:.3f}] {r['chunk_id']} ({r['source_type']})")
    print(f"\nGenerated answer:\n{result['answer']}")
    print(f"\nGold expected answer:\n{gold_answer}")
    print("=" * 100)


In [10]:
# q01: single-hop fact lookup (strongest genetic risk factor)
q = questions_by_id["q01"]
result = vectorrag_answer(q["question"], k=5)
show_result(result, q["expected_answer"])


Question: What is the strongest genetic risk factor for sporadic Alzheimer's disease?

Retrieved chunks:
  [0.716] pubmed_23276979__chunk000 (pubmed)
  [0.686] wikipedia_alzheimers_disease__chunk009 (wikipedia)
  [0.653] pubmed_34101789__chunk000 (pubmed)
  [0.642] textbook_statpearls_alzheimer_disease__chunk010 (textbook)
  [0.640] wikipedia_alzheimers_disease__chunk010 (wikipedia)

Generated answer:
The strongest genetic risk factor for sporadic Alzheimer's disease disease is APOE4 allele of the APOEε4 gene.

Gold expected answer:
The APOE epsilon-4 (e4) allele is the strongest genetic risk factor for sporadic Alzheimer's disease, confirmed by large-scale GWAS meta-analyses.


In [11]:
# q06: single-hop fact lookup (tacrine / hepatotoxicity)
q = questions_by_id["q06"]
result = vectorrag_answer(q["question"], k=5)
show_result(result, q["expected_answer"])


Question: What was the first cholinesterase inhibitor licensed for Alzheimer's treatment, and why is it no longer used?

Retrieved chunks:
  [0.623] wikipedia_cholinesterase_inhibitor__chunk001 (wikipedia)
  [0.616] textbook_statpearls_alzheimer_disease__chunk023 (textbook)
  [0.613] wikipedia_lecanemab__chunk006 (wikipedia)
  [0.610] wikipedia_cholinesterase_inhibitor__chunk000 (wikipedia)
  [0.604] wikipedia_memantine__chunk009 (wikipedia)

Generated answer:
The first cholinesterase inhibitor licensed for Alzheimer's treatment was tacastigmine, which was approved in the US for the treatment of Alzheimer's disease and's treatment.

Gold expected answer:
Tacrine was the first licensed cholinesterase inhibitor for Alzheimer's disease; it is no longer used because of its hepatotoxicity.


In [12]:
# q17: a multi-hop graphrag-favored question, to see how VectorRAG handles it by comparison
q = questions_by_id["q17"]
result = vectorrag_answer(q["question"], k=5)
show_result(result, q["expected_answer"])


Question: How does mild cognitive impairment connect to Alzheimer's disease and the biomarkers used to track its progression?

Retrieved chunks:
  [0.668] textbook_statpearls_alzheimer_disease__chunk033 (textbook)
  [0.634] wikipedia_mild_cognitive_impairment__chunk007 (wikipedia)
  [0.634] wikipedia_alzheimers_disease__chunk004 (wikipedia)
  [0.597] wikipedia_mild_cognitive_impairment__chunk003 (wikipedia)
  [0.590] wikipedia_mild_cognitive_impairment__chunk002 (wikipedia)

Generated answer:
Mild cognitive impairment (MCI) is a risk factor for dementia, and Alzheimer's disease. Biomarkers used to track its progression include low levels of amyloid and increased tau proteins in CSF, ApoE4 positivity, scores on cognitive tests like paired associates immediate recall test and symbol symbol substitution test, increased tau protein in CSF, and measurements of right entorhinal thickness and right entorhinal cortex and right hippocampal volume on MRI.

Gold expected answer:
Amnestic MCI is f

## 3.10 Discussion: VectorRAG's strengths and limitations

**Strengths:**
- Simple, fast, and robust -- no entity linking step that can silently fail to find
  the right node, unlike GraphRAG.
- Works well for questions whose answer is stated in (approximately) one place in the
  corpus -- semantic similarity search reliably surfaces the right passage even when
  the question is phrased very differently from the source text.
- Naturally surfaces supporting *evidence text*, not just a fact -- useful for
  citation/verification.

**Limitations:**
- No explicit relational structure: if an answer requires combining facts from two
  chunks that are semantically dissimilar from each other (e.g. a genetics chunk and a
  treatment chunk that never mention each other directly), nothing about the vector
  index tells the retriever they should be combined -- it can only retrieve chunks
  each similar to the *query*, not chunks that are relevant to each other.
- Long/broad synthesis questions (e.g. "compare treatment classes and their outcomes")
  may need supporting evidence spread across many chunks, more than a top-k=5
  retrieval can practically include in one prompt.
- Retrieval quality is entirely dependent on embedding model quality and how well the
  question's phrasing semantically overlaps with the source text's phrasing.

This is exactly the gap Notebook 2's GraphRAG (explicit relations, multi-hop
traversal) is meant to cover -- and exactly why Notebook 4 combines both rather than
picking one.

**Next:** [`04_hybridrag_and_evaluation.ipynb`](04_hybridrag_and_evaluation.ipynb)
combines this VectorRAG pipeline with Notebook 2's GraphRAG pipeline into a single
HybridRAG pipeline, then scores all three approaches against the full 30-question
evaluation set.
